<a href="https://colab.research.google.com/github/Akshatha7710/RAG-based-Document-Question-Answering-System-using-LangChain-FAISS-and-LLMs/blob/main/Question%20Answering%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG-Based Document Question Answering System v4
**Stack:** PyMuPDF · ChromaDB (persistent) · Sentence-Transformers · Gemini 2.5 Flash


In [1]:
# Cell 1: Install Dependencies
!pip install -q pymupdf chromadb sentence-transformers langchain-text-splitters \
               google-generativeai tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 660.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
# Cell 2: Set API Key
import os
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key or len(api_key) < 10:
    raise ValueError("❌ GEMINI_API_KEY secret is missing or looks invalid. "
                     "Add it via Colab Secrets (🔑 icon in the left sidebar).")

os.environ["GEMINI_API_KEY"] = api_key
print("✅ API Key loaded:", os.environ["GEMINI_API_KEY"][:10], "...")

✅ API Key loaded: AIzaSyDFX_ ...


In [3]:
# Cell 3: Imports & Setup
import fitz
import chromadb
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import time
import os

# ── Central configuration ──────────────────────────────────────────
CONFIG = {
    "model":        "gemini-2.5-flash",
    "embedder":     "all-MiniLM-L6-v2",
    "chunk_size":   500,
    "chunk_overlap":100,
    "top_k":        3,
    "db_path":      "/content/chroma_db",
    "max_retries":  5,
}

# ── Gemini ────────────────────────────────────────────────────────
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
llm = genai.GenerativeModel(CONFIG["model"])

# ── Embedding model (local, no API needed) ────────────────────────
embedder = SentenceTransformer(CONFIG["embedder"])

# ── Persistent vector database ────────────────────────────────────
# Mount Google Drive for true long-term persistence:
#   from google.colab import drive; drive.mount('/content/drive')
#   CONFIG['db_path'] = "/content/drive/MyDrive/rag_chroma_db"
chroma     = chromadb.PersistentClient(path=CONFIG["db_path"])
collection = chroma.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

# ── Text splitter ─────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
)

print(f"✅ All components initialized")
print(f"📂 Vector DB path: {CONFIG['db_path']}")
print(f"📄 Documents already in collection: {collection.count()}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All components initialized
📂 Vector DB path: /content/chroma_db
📄 Documents already in collection: 0


In [4]:
# Cell 4: Core Functions

# ── Retry helper ──────────────────────────────────────────────────
def call_gemini(prompt: str, max_attempts: int = CONFIG["max_retries"]) -> str:
    """Call Gemini with exponential backoff on 503 / quota errors."""
    for attempt in range(max_attempts):
        try:
            response = llm.generate_content(prompt)
            return response.text
        except Exception as e:
            if attempt == max_attempts - 1:
                return f"❌ Failed after {max_attempts} attempts: {e}"
            wait = 2 ** attempt * 5   # 5s, 10s, 20s, 40s …
            print(f"⚠️  Attempt {attempt+1} failed — retrying in {wait}s… ({e})")
            time.sleep(wait)


# ── Text extraction (PDF + TXT) ───────────────────────────────────
def extract_text(path: str) -> str:
    """Extract raw text from a PDF or plain-text file."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        doc  = fitz.open(path)
        text = "\n\n".join(page.get_text() for page in doc)
        doc.close()
        return text
    elif ext == ".txt":
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            return f.read()
    else:
        raise ValueError(f"Unsupported file type '{ext}'. Only .pdf and .txt are supported.")


# ── Ingestion (multi-document) ────────────────────────────────────
def ingest_file(path: str):
    """
    Extract text from a PDF or TXT file, chunk it, embed it, and store in ChromaDB.
    Each chunk is tagged with its source filename so answers can cite it.
    Safe to call multiple times — uses upsert with stable IDs.
    """
    filename = os.path.basename(path)
    text     = extract_text(path)

    chunks     = splitter.split_text(text)
    embeddings = embedder.encode(chunks, show_progress_bar=True).tolist()
    ids        = [f"{filename}_chunk_{i}" for i in range(len(chunks))]
    metadatas  = [{"source": filename, "chunk_index": i} for i in range(len(chunks))]

    collection.upsert(ids=ids, embeddings=embeddings, documents=chunks, metadatas=metadatas)
    print(f"✅ Ingested {len(chunks)} chunks from '{filename}'")


# Keep legacy alias
def ingest_pdf(path: str):
    ingest_file(path)


def ingest_multiple_files(paths: list):
    """Ingest a list of PDF or TXT file paths."""
    for path in tqdm(paths, desc="Ingesting files"):
        ingest_file(path)
    print(f"\n📚 Total chunks in DB: {collection.count()}")


# Keep legacy alias
def ingest_multiple_pdfs(paths: list):
    ingest_multiple_files(paths)


# ── Utility: list ingested sources ───────────────────────────────
def list_sources():
    """Print a summary of all documents currently in the vector store."""
    total = collection.count()
    if total == 0:
        print("⚠️  Vector store is empty. Upload and ingest documents first.")
        return
    all_meta = collection.get(include=["metadatas"])["metadatas"]
    sources  = {}
    for m in all_meta:
        src = m.get("source", "unknown")
        sources[src] = sources.get(src, 0) + 1
    print(f"📚 {total} total chunks from {len(sources)} file(s):")
    for src, count in sorted(sources.items()):
        print(f"   • {src}  ({count} chunks)")


# ── Empty-collection guard ────────────────────────────────────────
def _require_documents():
    if collection.count() == 0:
        raise RuntimeError(
            "❌ No documents in the vector store. "
            "Run Cell 5 to upload and ingest documents first."
        )


# ── Single-turn QA ────────────────────────────────────────────────
def ask(question: str, top_k: int = CONFIG["top_k"], show_sources: bool = True) -> str:
    """Retrieve relevant chunks and generate a grounded answer."""
    _require_documents()
    q_emb   = embedder.encode([question]).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=top_k,
        include=["documents", "distances", "metadatas"]
    )
    chunks    = results["documents"][0]
    scores    = results["distances"][0]
    metadatas = results["metadatas"][0]
    context   = "\n\n---\n\n".join(chunks)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say \"I don't have enough information to answer that.\"

CONTEXT:
{context}

QUESTION:
{question}"""

    answer = call_gemini(prompt)

    if show_sources:
        print("─" * 60)
        print("📄 SOURCE CHUNKS USED:")
        for i, (chunk, score, meta) in enumerate(zip(chunks, scores, metadatas)):
            similarity = round((1 - score) * 100, 1)
            print(f"\n[Source {i+1}] — {similarity}% match | file: {meta.get('source', 'unknown')}")
            print(chunk[:300] + "..." if len(chunk) > 300 else chunk)
        print("─" * 60)

    return answer


# ── Conversational chat ───────────────────────────────────────────
class RAGChat:
    """
    Stateful chat wrapper that keeps message history so follow-up
    questions ("Can you elaborate?", "What about X?") work correctly.
    """
    def __init__(self, top_k: int = CONFIG["top_k"], max_history_turns: int = 6):
        self.top_k = top_k
        self.max_history_turns = max_history_turns
        self.history: list = []

    def _retrieve(self, question: str):
        q_emb   = embedder.encode([question]).tolist()
        results = collection.query(
            query_embeddings=q_emb,
            n_results=self.top_k,
            include=["documents", "distances", "metadatas"]
        )
        chunks    = results["documents"][0]
        scores    = results["distances"][0]
        metadatas = results["metadatas"][0]
        context   = "\n\n---\n\n".join(chunks)
        return context, chunks, scores, metadatas

    def chat(self, user_message: str, show_sources: bool = False) -> str:
        _require_documents()
        context, chunks, scores, metadatas = self._retrieve(user_message)

        history_str = ""
        for turn in self.history[-(self.max_history_turns * 2):]:
            role = "User" if turn["role"] == "user" else "Assistant"
            history_str += f"{role}: {turn['content']}\n"

        prompt = f"""You are a helpful assistant. Answer using ONLY the context provided.
If the answer is not in the context, say \"I don't have enough information to answer that.\"
Use the conversation history to understand follow-up questions.

CONTEXT FROM DOCUMENTS:
{context}

CONVERSATION HISTORY:
{history_str}
User: {user_message}
Assistant:"""

        answer = call_gemini(prompt)

        self.history.append({"role": "user",      "content": user_message})
        self.history.append({"role": "assistant", "content": answer})

        if show_sources:
            print("─" * 60)
            print("📄 SOURCE CHUNKS USED:")
            for i, (chunk, score, meta) in enumerate(zip(chunks, scores, metadatas)):
                similarity = round((1 - score) * 100, 1)
                print(f"[Source {i+1}] {similarity}% | {meta.get('source', 'unknown')}")
            print("─" * 60)

        return answer

    def reset(self):
        """Clear conversation history."""
        self.history = []
        print("🔄 Conversation history cleared.")


print("✅ Functions ready")

✅ Functions ready


In [6]:
# Cell 5: Upload & Ingest One or More PDFs or TXT Files
from google.colab import files

uploaded = files.upload()   # opens file picker — select .pdf or .txt files
ingest_multiple_files(list(uploaded.keys()))
list_sources()

Saving ai_basics.pdf to ai_basics.pdf


Ingesting files:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Ingesting files: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]

✅ Ingested 13 chunks from 'ai_basics.pdf'

📚 Total chunks in DB: 13
📚 13 total chunks from 1 file(s):
   • ai_basics.pdf  (13 chunks)


In [7]:
# Cell 6: Single-Turn Q&A
questions = [
    "What are the three types of machine learning?",
    "How is AI used in healthcare?",
    "What are the ethical concerns around AI?",
    "What is the difference between deep learning and machine learning?"
]

for q in questions:
    print("\n" + "="*60)
    print(f"Q: {q}")
    print(ask(q))
    time.sleep(5)


Q: What are the three types of machine learning?
────────────────────────────────────────────────────────────
📄 SOURCE CHUNKS USED:

[Source 1] — 74.0% match | file: ai_basics.pdf
Machine Learning (ML) is a subset of AI that enables systems to learn and improve from experience
without being explicitly programmed. Instead of writing rules manually, ML algorithms learn patterns
from data.
There are three main types of machine learning:
- Supervised Learning: The model is traine...

[Source 2] — 57.9% match | file: ai_basics.pdf
Introduction to Artificial Intelligence
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems.
These processes include learning, reasoning, and self-correction. AI has become one of the most
transformative technologies of the 21st century, impacting indus...

[Source 3] — 53.9% match | file: ai_basics.pdf
- Unsupervised Learning: The model finds hidden patterns in unlabeled data. Clustering customers by
purchasing beha

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1721.18ms


────────────────────────────────────────────────────────────
📄 SOURCE CHUNKS USED:

[Source 1] — 64.6% match | file: ai_basics.pdf
Introduction to Artificial Intelligence
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems.
These processes include learning, reasoning, and self-correction. AI has become one of the most
transformative technologies of the 21st century, impacting indus...

[Source 2] — 63.8% match | file: ai_basics.pdf
Predictive analytics powered by AI can identify patients at risk of deterioration, enabling earlier
intervention. Virtual health assistants help patients manage chronic conditions and answer medical
questions around the clock.
6. Ethical Considerations in AI
As AI becomes more powerful, ethical conc...

[Source 3] — 63.2% match | file: ai_basics.pdf
transforming healthcare, finance, education, and many other sectors. While the opportunities are
immense, responsible development that addresses bias, privacy, and t

In [8]:
# Cell 7: Ask Your Own Single Question
my_question = "Summarize the document in 3 bullet points"  # ← change this

print(f"Q: {my_question}")
print(ask(my_question))

Q: Summarize the document in 3 bullet points
────────────────────────────────────────────────────────────
📄 SOURCE CHUNKS USED:

[Source 1] — 14.4% match | file: ai_basics.pdf
modern large language models like GPT and Claude.
3. Natural Language Processing
Natural Language Processing (NLP) is a branch of AI focused on enabling computers to understand
and generate human language. Applications include machine translation, sentiment analysis, chatbots,
and text summarization...

[Source 2] — 10.6% match | file: ai_basics.pdf
government bodies worldwide are working to establish frameworks for ethical AI.
7. The Future of AI
The future of AI holds enormous promise. Artificial General Intelligence (AGI), which would match
human-level reasoning across all domains, remains a long-term research goal. In the near term, we can
...

[Source 3] — 7.4% match | file: ai_basics.pdf
transforming healthcare, finance, education, and many other sectors. While the opportunities are
immense, responsible dev

In [9]:
# Cell 8: Scripted Conversational Chat Demo
bot = RAGChat(top_k=3)

turns = [
    "What are the three types of machine learning?",
    "Can you give me a real-world example of the second one?",
    "What about the ethical concerns related to that?",
]

for turn in turns:
    print("\n" + "="*60)
    print(f"👤 You: {turn}")
    reply = bot.chat(turn, show_sources=False)
    print(f"🤖 Bot: {reply}")
    time.sleep(5)


👤 You: What are the three types of machine learning?
🤖 Bot: The three main types of machine learning are:
- Supervised Learning
- Unsupervised Learning
- Reinforcement Learning

👤 You: Can you give me a real-world example of the second one?
🤖 Bot: Clustering customers by purchasing behavior is one example.

👤 You: What about the ethical concerns related to that?
🤖 Bot: Key ethical concerns include algorithmic bias, privacy concerns around data collection and surveillance, and the challenge of making AI systems explainable and transparent.


In [10]:
# Cell 9: ✨ Interactive Chat Loop (v4)
#
# Run this cell and type your questions live in the input box.
# Full conversation memory is kept across turns — follow-up questions work.
#
# Commands:
#   /help             — show available commands
#   quit  or  exit    — stop the loop
#   reset             — clear conversation history and start fresh
#   history           — print the conversation so far
#   sources on/off    — toggle whether retrieved chunks are shown
#   list sources      — show all ingested documents

live_bot     = RAGChat(top_k=CONFIG["top_k"])
show_sources = False

HELP_TEXT = """
📖 Available commands:
  /help          — show this message
  quit / exit    — end the session
  reset          — clear chat history
  history        — print conversation so far
  sources on     — show retrieved chunks after each answer
  sources off    — hide retrieved chunks
  list sources   — list all ingested documents
"""

print("💬 RAG Chat ready! Type your question below.")
print("   Type /help for a list of commands.")
print("=" * 60)

while True:
    try:
        user_input = input("\n👤 You: ").strip()
    except EOFError:
        print("\n[Non-interactive environment detected — exiting chat loop]")
        break

    if not user_input:
        continue

    # ── Built-in commands ─────────────────────────────────────────────
    cmd = user_input.lower()

    if cmd in ("/help", "help"):
        print(HELP_TEXT)

    elif cmd in ("quit", "exit"):
        print("👋 Ending chat session.")
        break

    elif cmd == "reset":
        live_bot.reset()

    elif cmd == "history":
        if not live_bot.history:
            print("[No history yet]")
        for turn in live_bot.history:
            role  = "👤 You" if turn["role"] == "user" else "🤖 Bot"
            print(f"{role}: {turn['content']}")

    elif cmd == "sources on":
        show_sources = True
        print("[Sources will now be shown]")

    elif cmd == "sources off":
        show_sources = False
        print("[Sources hidden]")

    elif cmd == "list sources":
        list_sources()

    else:
        # ── Normal question ───────────────────────────────────────────
        reply = live_bot.chat(user_input, show_sources=show_sources)
        print(f"🤖 Bot: {reply}")

💬 RAG Chat ready! Type your question below.
   Type /help for a list of commands.

👤 You: What are the three types of machine learning?
🤖 Bot: The three main types of machine learning are:
- Supervised Learning
- Unsupervised Learning
- Reinforcement Learning

👤 You: quit
👋 Ending chat session.


In [11]:
# Cell 10: Evaluation — Semantic Similarity Scoring
import numpy as np
import csv

eval_set = [
    {
        "question": "What are the three types of machine learning?",
        "expected": "The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning."
    },
    {
        "question": "How is AI used in healthcare?",
        "expected": "AI is used in healthcare for predictive analytics to identify at-risk patients and for virtual health assistants that help manage chronic conditions."
    },
    {
        "question": "What is deep learning?",
        "expected": "Deep learning is a subset of machine learning that uses neural networks with many layers to model complex patterns."
    },
]

def semantic_similarity(a: str, b: str) -> float:
    vecs = embedder.encode([a, b])
    return float(np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1])))


def score_label(s: float) -> str:
    if s >= 0.90: return "🟢 Excellent"
    if s >= 0.80: return "🟡 Good"
    if s >= 0.70: return "🟠 Fair"
    return "🔴 Poor"


scores  = []
rows    = []
print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

for item in eval_set:
    predicted = ask(item["question"], show_sources=False)
    score     = semantic_similarity(predicted, item["expected"])
    scores.append(score)
    rows.append({"question": item["question"], "expected": item["expected"],
                 "predicted": predicted, "score": round(score, 4)})

    print(f"\nQ:        {item['question']}")
    print(f"Expected: {item['expected'][:120]}")
    print(f"Got:      {predicted[:120]}")
    print(f"Score:    {score:.3f}  {score_label(score)}")
    time.sleep(5)

avg = np.mean(scores)
print("\n" + "=" * 60)
print(f"Average semantic similarity: {avg:.3f}  {score_label(avg)}")
print("(1.0 = perfect match, ≥0.90 Excellent, ≥0.80 Good, ≥0.70 Fair)")

# ── Export results to CSV ────────────────────────────────────────
csv_path = "/content/eval_results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "expected", "predicted", "score"])
    writer.writeheader()
    writer.writerows(rows)
print(f"\n💾 Results saved to {csv_path}")
try:
    from google.colab import files as colab_files
    colab_files.download(csv_path)
except Exception:
    pass

EVALUATION RESULTS

Q:        What are the three types of machine learning?
Expected: The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.
Got:      The three main types of machine learning are:
- Supervised Learning
- Unsupervised Learning
- Reinforcement Learning
Score:    0.972  🟢 Excellent

Q:        How is AI used in healthcare?
Expected: AI is used in healthcare for predictive analytics to identify at-risk patients and for virtual health assistants that he
Got:      Predictive analytics powered by AI can identify patients at risk of deterioration, enabling earlier intervention. Virtua
Score:    0.931  🟢 Excellent

Q:        What is deep learning?
Expected: Deep learning is a subset of machine learning that uses neural networks with many layers to model complex patterns.
Got:      Deep Learning is a subset of machine learning that uses neural networks with many layers (hence deep) to model complex p
Score:    0.991  🟢 Exce

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>